# RAW to RGB — Part 1: Conversion and Normalization

In this notebook we:
1. Read a Canon CR2 RAW file as a raw Bayer array using `rawpy`
2. Determine the **black level** from a dark frame
3. Determine the **clipping point** from an overexposed frame
4. Normalize the image to the range 0…1

**Required files in the same folder as this notebook:**
- `IMG_2358.CR2` — black frame (lens cap on)
- `IMG_2364.CR2` — correctly exposed frame
- `IMG_2367.CR2` — overexposed frame (clipped whites)

> **Note on rawpy:** Install with `pip install rawpy`. It wraps LibRaw directly —
> no external binary needed, unlike the MATLAB workflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rawpy
import os

## sRGB transfer functions

We define these once and reuse them throughout all notebooks.

$$\text{linear} \rightarrow \text{sRGB}: \quad
V = \begin{cases} 1.055\, L^{1/2.4} - 0.055 & L > 0.0031308 \\ 12.92\, L & \text{otherwise} \end{cases}$$

In [ ]:
linear2sRGB = lambda x: np.where(x > 0.0031308, 1.055 * np.clip(x, 0, None) ** (1/2.4) - 0.055, 12.92 * x)
sRGB2linear = lambda x: np.where(x > 0.04045, ((x + 0.055) / 1.055) ** 2.4, x / 12.92)

## Load RAW files

We use `rawpy` with `output_color=rawpy.ColorSpace.raw` and `no_auto_bright=True`
to get the unprocessed Bayer data as a 2D array.
The `postprocess` call returns a 16-bit image; we take only the first channel
because at this stage we want the raw sensor values, not a debayered colour image.

A cleaner approach is `rawpy.imread(...).raw_image_visible` which gives the
Bayer plane directly as a 2D uint16 array — that is what we use here.

In [ ]:
image_folder = '.'   # adjust if your CR2 files are in a different folder

def load_raw_bayer(path):
    """Return the visible Bayer plane as a float64 array (uint16 values)."""
    with rawpy.imread(path) as raw:
        return raw.raw_image_visible.astype(np.float64)

black_frame      = load_raw_bayer(os.path.join(image_folder, 'IMG_2358.CR2'))
img              = load_raw_bayer(os.path.join(image_folder, 'IMG_2364.CR2'))
overexposed_frame = load_raw_bayer(os.path.join(image_folder, 'IMG_2367.CR2'))

print(f'Image shape : {img.shape}  dtype: {img.dtype}')
print(f'Value range : {img.min():.0f} … {img.max():.0f}')

## First look — apply sRGB gamma so the image is visible

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(linear2sRGB(img / 65535), cmap='gray')
ax.set_title('Raw Bayer image with sRGB gamma — still looks dark and monochrome')
ax.axis('off')
plt.tight_layout()
plt.show()

## Determine the black level

The black frame (lens cap on) reveals the sensor's dark current and read noise.
We use the **median** rather than the mean because it is more robust against hot pixels.

In [ ]:
print(f'Black frame  min    : {black_frame.min():.1f}')
print(f'Black frame  mean   : {black_frame.mean():.1f}')
print(f'Black frame  median : {np.median(black_frame):.1f}')
print(f'Black frame  max    : {black_frame.max():.1f}')

black_level = np.median(black_frame)
print(f'\nBlack level  = {black_level:.1f}')

## Measure read noise standard deviation

We crop away the border (optical black pixels) before measuring noise.
The standard deviation of the black frame tells us how much noise the sensor adds
even when no light falls on it.

In [ ]:
crop = 600
black_crop = black_frame[crop:-crop, crop:-crop]
black_noise_std = np.std(black_crop)
print(f'Read noise std dev: {black_noise_std:.2f} code values')

# Visualize the black frame after subtracting the black level
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(linear2sRGB(np.clip((black_frame - black_level) / 200, 0, 1)), cmap='gray')
axes[0].set_title('Black frame − black_level')
axes[0].axis('off')

axes[1].imshow(linear2sRGB(np.clip((black_frame - black_level + 3*black_noise_std) / 200, 0, 1)), cmap='gray')
axes[1].set_title('Black frame − black_level + 3σ  (preserves noise floor)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Determine the clipping point

The overexposed frame shows us the maximum code value the sensor produces before clipping.
Again we use the median for robustness.

In [ ]:
print(f'Overexposed frame  min    : {overexposed_frame.min():.1f}')
print(f'Overexposed frame  mean   : {overexposed_frame.mean():.1f}')
print(f'Overexposed frame  median : {np.median(overexposed_frame):.1f}')
print(f'Overexposed frame  max    : {overexposed_frame.max():.1f}')
print(f'Overexposed frame  std    : {np.std(overexposed_frame):.1f}')

clipping_point = np.median(overexposed_frame)
print(f'\nClipping point = {clipping_point:.1f}')

## Normalize to 0…1

We subtract the black level and divide by the dynamic range:

$$\hat{I} = \frac{I - \text{blackLevel}}{\text{clippingPoint} - \text{blackLevel}}$$

The `stops_above_white` parameter lets us adjust exposure in powers of 2 (stops).

In [ ]:
img_black_subtracted = (img - black_level) / (clipping_point - black_level)

stops_above_white = 0   # try 1, 2, 3 to brighten

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(linear2sRGB(np.clip(img_black_subtracted * 2**stops_above_white, 0, 1)), cmap='gray')
ax.set_title(f'Black-subtracted and normalized  (exposure +{stops_above_white} stops)')
ax.axis('off')
plt.tight_layout()
plt.show()

## Save result for the next notebook

In [ ]:
# Normalize black_noise_std to the same scale as the image
black_noise_std_norm = black_noise_std / (clipping_point - black_level)

np.savez('20_RawBayerImage_BlackSubtracted.npz',
         img_black_subtracted=img_black_subtracted,
         black_noise_std=black_noise_std_norm)

print('Saved: 20_RawBayerImage_BlackSubtracted.npz')